In [0]:
%run ./ngo_medallion_Mohamed_Al_Sharqawy/00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- 1. تنظيف جدول الحالات (Cases) ---
cases_bronze = spark.table(bronze_cases)

cases_silver = (
    cases_bronze
    .filter(F.col("case_id").isNotNull())
    .withColumn("created_date", F.to_date("created_date")) 
)

# إزالة التكرار (Deduplication)
w_case = Window.partitionBy("case_id").orderBy(F.col("_ingested_at").desc())
cases_silver = (cases_silver.withColumn("rn", F.row_number().over(w_case))
                .filter(F.col("rn") == 1)
                .drop("rn"))

cases_silver.write.format("delta").mode("overwrite").saveAsTable(silver_cases)


# --- 2. تنظيف جدول الخدمات (Services) ---
services_bronze = spark.table(bronze_services)

services_silver = (
    services_bronze
    .filter(F.col("service_id").isNotNull())
    # استخدام actual_date وتحويله إلى تاريخ باسم service_date
    .withColumn("service_date", F.to_date(F.col("actual_date")))
)

# إزالة التكرار
w_srv = Window.partitionBy("service_id").orderBy(F.col("_ingested_at").desc())
services_silver = (services_silver.withColumn("rn", F.row_number().over(w_srv))
                   .filter(F.col("rn") == 1)
                   .drop("rn"))

services_silver.write.format("delta").mode("overwrite").saveAsTable(silver_services)


# --- 3. فحوصات الجودة المؤتمتة (Automated Quality Checks) ---
checks_data = [
    ("null_case_ids", spark.table(silver_cases).filter("case_id IS NULL").count()),
    ("duplicate_cases", spark.table(silver_cases).groupBy("case_id").count().filter("count > 1").count()),
    ("null_service_ids", spark.table(silver_services).filter("service_id IS NULL").count()),
    ("duplicate_services", spark.table(silver_services).groupBy("service_id").count().filter("count > 1").count())
]

quality_df = (
    spark.createDataFrame(checks_data, ["check_name", "failed_count"])
    .withColumn("status", F.when(F.col("failed_count") == 0, "PASSED").otherwise("FAILED"))
    .withColumn("checked_at", F.current_timestamp())
)

quality_df.write.format("delta").mode("overwrite").saveAsTable(gold_quality_summary)
display(quality_df)

check_name,failed_count,status,checked_at
null_case_ids,0,PASSED,2026-08-11T20:36:39.807Z
duplicate_cases,0,PASSED,2026-08-11T20:36:39.807Z
null_service_ids,0,PASSED,2026-08-11T20:36:39.807Z
duplicate_services,0,PASSED,2026-08-11T20:36:39.807Z
